In [1]:
import sys
import os
import subprocess

print("Python executable:", sys.executable)
print("Python prefix:", sys.prefix)
print("Conda environment:", os.environ.get('CONDA_DEFAULT_ENV'))
print("Conda prefix:", os.environ.get('CONDA_PREFIX'))

# Check pip location
result = subprocess.run(['which', 'pip'], capture_output=True, text=True)
print("Pip location:", result.stdout.strip())

# Check pip version
result = subprocess.run(['pip', '--version'], capture_output=True, text=True)
print("Pip version:", result.stdout.strip())

# Add environment's bin directory to PATH
env_bin = os.path.dirname(sys.executable)
current_path = os.environ['PATH']
if env_bin not in current_path:
    os.environ['PATH'] = f"{env_bin}:{current_path}"

# Verify fix
result = subprocess.run(['which', 'pip'], capture_output=True, text=True)
print("Fixed pip location:", result.stdout.strip())

Python executable: /home/sagemaker-user/.conda/envs/venv-LLMsFromScratch/bin/python3
Python prefix: /home/sagemaker-user/.conda/envs/venv-LLMsFromScratch
Conda environment: base
Conda prefix: /opt/conda
Pip location: /opt/conda/bin/pip
Pip version: pip 25.1.1 from /opt/conda/lib/python3.12/site-packages/pip (python 3.12)
Fixed pip location: /home/sagemaker-user/.conda/envs/venv-LLMsFromScratch/bin/pip


In [3]:
!pwd

/home/sagemaker-user/LLMsFromScratch/Chapter_2


# Code

In [4]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total Characters:", len(raw_text))

Total Characters: 20479


In [5]:
print(raw_text[:99])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


### Split text

In [6]:
import re
text = "Kya haal chal bro? sab sahi hai?"

result = re.split(r'(\s)', text)

print(result)

['Kya', ' ', 'haal', ' ', 'chal', ' ', 'bro?', ' ', 'sab', ' ', 'sahi', ' ', 'hai?']


In [9]:
# spliting punctuation too

result = re.split(r'([,.?]|\s)', text)

print(result)

['Kya', ' ', 'haal', ' ', 'chal', ' ', 'bro', '?', '', ' ', 'sab', ' ', 'sahi', ' ', 'hai', '?', '']


In [10]:
# removing spaces

result = [ item for item in result if item.strip() ]

print(result)

['Kya', 'haal', 'chal', 'bro', '?', 'sab', 'sahi', 'hai', '?']


In [11]:
# test it all 

text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]

print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [16]:
# applying the tokenizer to complete text file

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)

preprocessed = [item.strip() for item in preprocessed if item.strip()]

print(len(preprocessed))

4690


In [17]:
# I first got around 9000 token - forgot removed the spaces.
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


### Converting to Token ID

In [26]:
# counts the unique token 
all_words = sorted(set(preprocessed))

vocab_size = len(all_words)

vocab_size

1130

In [54]:
# Creating a vocabulary

vocab = { token:integer for integer,token in enumerate(all_words) }

# vocab 

for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


### Implementing a simple text tokenizer

In [27]:
'''
#1 Stores the vocabulary as a class attribute for access in the encode and decode methods
#2 Creates an inverse vocabulary that maps token IDs back to the original text tokens
#3 Processes input text into token IDs
#4 Converts token IDs back into text
#5 Removes spaces before the specified punctuation 
'''

class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items() }

    def encode(self,text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [ item.strip() for item in preprocessed if item.strip() ]
        ids = [self.str_to_int[s] for s in preprocessed]

        return ids

    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)

        return text

In [28]:
# calling the function 

tokenizer = SimpleTokenizerV1(vocab) # passing the mapping of word to token id

text =  """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)

print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [29]:
# Printing back the words from token id - this is what happens at the end of LLM output just before you see the output

print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [31]:
# calling the function on words that are not stored in the tokenizer dicitionary - common key value error

hinid_test_tokenizer = SimpleTokenizerV1(vocab) # passing the mapping of word to token id

text =  """"kya haal chal bhai"""

ids = hinid_test_tokenizer.encode(text)

print(ids)


KeyError: 'kya'

### Adding Custom Token

In [55]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}             

print(len(vocab.items()))

1132


In [56]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


### Simple text tokenizer that handles unknown words

In [62]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [ item.strip() for item in preprocessed if item.strip() ]
        preprocessed = [ item if item in self.str_to_int else "<|unk|>" for item in preprocessed]

        ids = [self.str_to_int[s] for s in preprocessed]
        
        return ids

    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [63]:
# this is how they combine large document to tell the model that this is sepearate doc

text1 = "Hello, do you like tea?"

text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [64]:
tokenizerv2 = SimpleTokenizerV2(vocab)
print(tokenizerv2.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [65]:
print(tokenizerv2.decode(tokenizerv2.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


### Byte pair encoding

In [69]:
%%capture
!pip install tiktoken

In [72]:
# Version 
from importlib.metadata import version
import tiktoken

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [73]:
bpe_tokenizer = tiktoken.get_encoding("gpt2")

In [75]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = bpe_tokenizer.encode(text, allowed_special= {"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [82]:
print(bpe_tokenizer.decode(integers))

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [89]:
text = "Swapnil"

encode_swapnil = bpe_tokenizer.encode(text, allowed_special= {"<|endoftext|>"})

encode_swapnil

[10462, 499, 45991]

In [96]:
print(bpe_tokenizer.decode([45991]))

nil


### Data sampling with a sliding window

In [110]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = bpe_tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [111]:
# removing first 50 token from passage - optional 
enc_sample = enc_text[50:]

In [120]:
# Creating context size - The context size determines how many tokens are included in the input. 
context_size = 5

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print("x: ", x)
print("y:      ", y)  

x:  [290, 4920, 2241, 287, 257]
y:       [4920, 2241, 287, 257, 4489]


In [122]:
for i in range(1, context_size):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "----->", desired)

[290] -----> 4920
[290, 4920] -----> 2241
[290, 4920, 2241] -----> 287
[290, 4920, 2241, 287] -----> 257


In [125]:
for i in range(1, context_size):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(bpe_tokenizer.decode(context), "----->", bpe_tokenizer.decode([desired]))

 and ----->  established
 and established ----->  himself
 and established himself ----->  in
 and established himself in ----->  a


In [ ]:
import torch

from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):

    def __init__(self, txt, tokenizer, max_length, stride):

        self.input_ids = []

        self.target_ids = []



        token_ids = tokenizer.encode(txt)    #1



        for i in range(0, len(token_ids) - max_length, stride):     #2

            input_chunk = token_ids[i:i + max_length]

            target_chunk = token_ids[i + 1: i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))

            self.target_ids.append(torch.tensor(target_chunk))



    def __len__(self):    #3

        return len(self.input_ids)



    def __getitem__(self, idx):         #4

        return self.input_ids[idx], self.target_ids[idx]